# Stage 3 Track B — cetacean detector v2 (prawn frames added as negatives)

**Why.** Detector v1 never saw a prawn frame in training, and in the end-to-end test it called 12 of 32 prawn frames cetacean
(prawn right answers fell from 27 to 20). v2 teaches it what prawn frames look like.

**Where the prawn pictures come from - and why the test stays clean**
- The only prawn frames are the 82 in `Stage_03/Test_data/PRAWNS` (plus 25 Stage 2 copies of some of them).
- The end-to-end audit already sets aside about 50 of the 82, because they are copies of pictures Track A / Stage 2 were developed on.
  **Those already-used frames become v2's prawn training and validation negatives.** They could never be test frames anyway.
- The remaining prawn frames, and anything that looks like them, stay untouched as **test** frames.
- Prawn frames are grouped at a strict similarity (0.95): if any picture in a group was already used, the whole group goes to training.

**What stays the same as v1:** the cetacean, tuna and sea train / val / test split is copied exactly from v1's file, so v2 is
fine-tuned from v1's weights without any leak, and the end-to-end test uses the same 61 cetacean photos as before (a fair v1-vs-v2 comparison).

**Other changes:** the confidence cut-off is chosen on val with false alarms averaged **per kind** (prawn counts as much as sea or tuna),
and prawn training frames are repeated (`PRAWN_OVERSAMPLE`) because there are few of them.

**Runs on your computer** (CPU is fine: fine-tuning needs far fewer epochs than v1). Set `TRIAL_EPOCHS = 1` first to time an epoch.
Saves `stage3_output/cetacean_detector_v2.pt` and `cetacean_detector_v2.json`.

In [1]:
# --- Cell 1: Settings ---
import os, sys, json, random, shutil, time
# Windows fix: PyTorch and NumPy can each bring their own copy of Intel's OpenMP library (libiomp5md.dll); loading both kills the
# kernel with "OMP: Error #15". Allowing the duplicate and loading PyTorch FIRST avoids it. Must run before numpy / torch are imported,
# so after a crash: restart the kernel and run from this cell.
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
try:
    import torch                     # loaded before numpy on purpose
except ImportError:
    torch = None
from pathlib import Path
import numpy as np
from PIL import Image

PROJECT_ROOT = Path(r"C:\Users\bcura\AI PostGrad Project\Saqqara-Marine-Anomaly-Detection")
S3 = PROJECT_ROOT / "Stage_03"
OUT_DIR = S3 / "stage3_output"
WORK_DIR = S3 / "cetacean_yolo_v2"
CET_LABELS = S3 / "Stage3_TrackB" / "train" / "labels"

V1_INFO = OUT_DIR / "cetacean_detector_v1.json"
V1_WEIGHTS = OUT_DIR / "cetacean_detector_v1.pt"

# Prawn pictures (every prawn frame we have)
PRAWN_POOL_DIRS = [S3 / "Test_data" / "PRAWNS", PROJECT_ROOT / "Stage_02" / "prawn_annotation_training" / "images" / "default"]
# What Track A / Stages 1-2 were developed on - the same list the end-to-end audit uses. A prawn frame that looks like any of
# these (or like anything detector v1 trained or tuned on) is 'already used' and may be trained on.
DEV_DIRS = [
    PROJECT_ROOT / "Stage_02" / "prawn_annotation_training" / "images" / "default",
    PROJECT_ROOT / "Stage_02" / "tuna_annotation_training" / "images" / "default",
    PROJECT_ROOT / "Stage_01" / "data" / "normal_sea" / "heldout_split",
    PROJECT_ROOT / "Stage_01" / "data" / "normal_sea" / "baseline_split",
    S3 / "Val_data" / "TUNA",
    S3 / "Val_data" / "SEA",
]
TEST_DATA_DIRS = {"tuna": S3 / "Test_data" / "TUNA", "sea": S3 / "Test_data" / "SEA"}   # extra false-alarm check (not trained on)

USED_SIMILARITY = 0.95          # prawn frame this close to a development picture = already used (same as the end-to-end audit)
PRAWN_GROUP_SIMILARITY = 0.95   # prawn frames this close to each other share a group (strict on purpose)
PRAWN_VAL_SHARE = 0.30          # of the already-used prawn groups, this share goes to val (to choose the cut-off), the rest to train
PRAWN_OVERSAMPLE = 5            # each prawn training picture is used this many times (YOLO augments each copy differently)
SEED = 42

FINETUNE_FROM_V1 = True         # start from v1's weights (needs v1's split, which is copied below)
FINETUNE_EPOCHS = 20
PATIENCE = 8
LR0 = 0.002                     # lower than training from scratch, so v1's cetacean skill is kept
BATCH = 16
TRIAL_EPOCHS = None             # e.g. 1 to time one epoch first
CONF_GRID = [0.10, 0.15, 0.20, 0.25, 0.30, 0.40, 0.50, 0.60, 0.70]

random.seed(SEED); np.random.seed(SEED)
IMG_EXTS = {".jpg", ".jpeg", ".png"}
def list_images(folder):
    folder = Path(folder)
    return sorted(p for p in folder.glob("*") if p.suffix.lower() in IMG_EXTS) if folder.exists() else []

for p in [V1_INFO, V1_WEIGHTS, CET_LABELS]:
    if not p.exists():
        raise RuntimeError(f"Not found: {p}\nRun Stage3_TrackB_Cetacean_Detector_v1 first, or fix PROJECT_ROOT.")
V1 = json.load(open(V1_INFO))
IMG_SIZE = int(V1.get("img_size", 352))
print("v1 cut-off:", V1["confidence_cutoff"], "| image size:", IMG_SIZE)

v1 cut-off: 0.1 | image size: 352


In [2]:
# --- Cell 2: Copy v1's split exactly (cetacean, tuna, sea) ---
def read_yolo(txt):
    rows = []
    for line in Path(txt).read_text().strip().splitlines():
        parts = line.split()
        if len(parts) >= 5:
            rows.append(tuple(float(v) for v in parts[1:5]))
    return rows

label_of = {p.stem: p for p in CET_LABELS.glob("*.txt")}
items, missing = [], 0
for split in ("train", "val", "test"):
    for e in V1["files"][split]:
        p = Path(e["source"])
        if not p.exists():
            missing += 1
            continue
        boxes = read_yolo(label_of[p.stem]) if e["kind"] == "cetacean" and p.stem in label_of else []
        items.append({"path": p, "kind": e["kind"], "split": split, "boxes": boxes, "group": f"v1_{e['group']}"})
print(f"Copied {len(items)} pictures from v1's split ({missing} listed files not found)")
print(f"{'kind':<10}" + "".join(f"{s:>8}" for s in ("train", "val", "test")))
for k in sorted({r["kind"] for r in items}):
    print(f"{k:<10}" + "".join(f"{sum(r['kind'] == k and r['split'] == s for r in items):>8}" for s in ("train", "val", "test")))

Copied 5867 pictures from v1's split (0 listed files not found)
kind         train     val    test
cetacean      4335     413     442
sea            244      62      60
tuna           225      50      36


In [3]:
# --- Cell 3: Prawn frames - already-used ones become training / val negatives, the rest stay as test ---
def thumb(path):
    with Image.open(path) as im:
        im.draft("L", (128, 128))
        g = np.asarray(im.convert("L").resize((32, 32), Image.BILINEAR), dtype=np.float32)
    def norm(a):
        a = a - a.mean(); n = np.linalg.norm(a)
        return (a / n).ravel() if n > 1e-6 else np.zeros(a.size, np.float32)
    return np.stack([norm(g), norm(g[:, ::-1]), norm(g[::-1, :]), norm(g[::-1, ::-1])])   # 4 orientations

def best_sim(A4, B):
    """For each picture in A (4 orientations each), the best similarity to any picture in B (upright)."""
    out = np.full(len(A4), -1.0)
    for s in range(0, len(B), 2048):
        Bb = B[s:s + 2048]
        out = np.maximum(out, np.max(np.einsum("nkd,md->nkm", A4, Bb), axis=(1, 2)))
    return out

t0 = time.time()
pool = list(dict.fromkeys(p for d in PRAWN_POOL_DIRS for p in list_images(d)))
P4 = np.stack([thumb(p) for p in pool])
print(f"Prawn pictures: {len(pool)}  (" + ", ".join(f"{len(list_images(d))} in {Path(d).parent.name}/{Path(d).name}" for d in PRAWN_POOL_DIRS) + ")")

# development pictures = Track A / Stage 1-2 folders + everything v1 trained or tuned on
dev_paths = list(dict.fromkeys([p for d in DEV_DIRS for p in list_images(d)] + [r["path"] for r in items if r["split"] in ("train", "val")]))
D = np.stack([thumb(p)[0] for p in dev_paths])
dev_names = {p.name for p in dev_paths}
sim_dev = best_sim(P4, D)
used = (sim_dev >= USED_SIMILARITY) | np.array([p.name in dev_names for p in pool])
print(f"Already used (look like a development picture, or same file name): {int(used.sum())} of {len(pool)}  ({time.time() - t0:.0f}s)")

# group prawn frames (strict); a group with ANY used picture goes to training
parent = list(range(len(pool)))
def find(i):
    while parent[i] != i:
        parent[i] = parent[parent[i]]; i = parent[i]
    return i
S = np.max(np.einsum("nkd,md->nkm", P4, P4[:, 0, :]), axis=1)
for a, b in zip(*np.nonzero(S >= PRAWN_GROUP_SIMILARITY)):
    if a != b:
        ra, rb = find(int(a)), find(int(b))
        if ra != rb:
            parent[rb] = ra
grp = [find(i) for i in range(len(pool))]
used_groups = {g for g, u in zip(grp, used) if u}
train_groups = sorted(used_groups)
random.Random(SEED).shuffle(train_groups)
n_val = max(1, int(round(PRAWN_VAL_SHARE * len(train_groups)))) if len(train_groups) > 1 else 0
val_groups = set(train_groups[:n_val])

n_new = {"train": 0, "val": 0, "test": 0}
for p, g in zip(pool, grp):
    split = "test" if g not in used_groups else ("val" if g in val_groups else "train")
    items.append({"path": p, "kind": "prawn", "split": split, "boxes": [], "group": f"prawn_{g}"})
    n_new[split] += 1
print(f"Prawn groups: {len(set(grp))} ({len(used_groups)} already used)")
print(f"Prawn pictures -> train {n_new['train']}, val {n_new['val']}, test {n_new['test']}")
n_test_td = sum(1 for r in items if r["kind"] == "prawn" and r["split"] == "test" and "Test_data" in str(r["path"]))
print(f"  prawn TEST frames from Test_data: {n_test_td} (the end-to-end test will use these, minus any its own audit sets aside)")
if n_new["test"] == 0:
    print("WARNING: no prawn frames are left for testing. The prawn result in the end-to-end test will not be meaningful.")
if n_new["val"] == 0:
    print("WARNING: no prawn frames in val - the cut-off cannot take prawns into account.")

Prawn pictures: 107  (82 in Test_data/PRAWNS, 25 in images/default)
Already used (look like a development picture, or same file name): 75 of 107  (41s)
Prawn groups: 38 (21 already used)
Prawn pictures -> train 48, val 27, test 32
  prawn TEST frames from Test_data: 32 (the end-to-end test will use these, minus any its own audit sets aside)


In [4]:
# --- Cell 4: Write the YOLO dataset ---
if WORK_DIR.exists():
    shutil.rmtree(WORK_DIR)
for s in ("train", "val", "test"):
    (WORK_DIR / "images" / s).mkdir(parents=True)
    (WORK_DIR / "labels" / s).mkdir(parents=True)
for k, r in enumerate(items):
    copies = PRAWN_OVERSAMPLE if (r["kind"] == "prawn" and r["split"] == "train") else 1
    names = []
    for c in range(copies):
        name = f"{r['kind']}_{k:06d}{'' if c == 0 else f'_dup{c}'}{Path(r['path']).suffix.lower()}"
        shutil.copy2(r["path"], WORK_DIR / "images" / r["split"] / name)
        lines = [f"0 {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}" for xc, yc, w, h in r["boxes"]]
        (WORK_DIR / "labels" / r["split"] / (Path(name).stem + ".txt")).write_text("\n".join(lines))
        names.append(name)
    r["yolo_name"] = names[0]
yaml_path = WORK_DIR / "data.yaml"
yaml_path.write_text(f"path: {WORK_DIR.as_posix()}\ntrain: images/train\nval: images/val\ntest: images/test\nnames:\n  0: cetacean\n")
for s in ("train", "val", "test"):
    print(f"  {s:<5} {len(list((WORK_DIR / 'images' / s).glob('*'))):>6} image files")
print("Dataset written to", WORK_DIR)

  train   5044 image files
  val      552 image files
  test     570 image files
Dataset written to C:\Users\bcura\AI PostGrad Project\Saqqara-Marine-Anomaly-Detection\Stage_03\cetacean_yolo_v2


In [5]:
# --- Cell 5: Fine-tune from v1 ---
try:
    import ultralytics
except ImportError:
    os.system(f"{sys.executable} -m pip install -q ultralytics")
    import ultralytics
from ultralytics import YOLO
import torch
GPU = torch.cuda.is_available()
print("ultralytics", ultralytics.__version__, "| GPU:", torch.cuda.get_device_name(0) if GPU else "none (CPU)")

EPOCHS = TRIAL_EPOCHS or FINETUNE_EPOCHS
if TRIAL_EPOCHS:
    print(f"TRIAL RUN: {EPOCHS} epoch(s) only - for timing; results will not be final.")
model = YOLO(str(V1_WEIGHTS) if FINETUNE_FROM_V1 else "yolov8n.pt")
train_res = model.train(data=str(yaml_path), imgsz=IMG_SIZE, epochs=EPOCHS, patience=PATIENCE, batch=BATCH, seed=SEED,
                        lr0=LR0 if FINETUNE_FROM_V1 else 0.01, warmup_epochs=1 if FINETUNE_FROM_V1 else 3,
                        project=str(WORK_DIR / "runs"), name="cetacean_v2", exist_ok=True,
                        device=0 if GPU else "cpu", workers=8 if GPU else 0, plots=True, verbose=False)
BEST = Path(train_res.save_dir) / "weights" / "best.pt"
print("Best weights:", BEST)

ultralytics 8.4.178 | GPU: none (CPU)
Ultralytics 8.4.178  Python-3.14.6 torch-2.14.0+cpu CPU (AMD Ryzen 7 7435HS)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Users\bcura\AI PostGrad Project\Saqqara-Marine-Anomaly-Detection\Stage_03\cetacean_yolo_v2\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=352, iou=0.7, kobj=1.0, line_width=None, lr0=0.002, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=C:\Users\bcura

In [6]:
# --- Cell 6: Predict val, test and the extra Test_data tuna / sea frames (v2, and v1 for comparison) ---
det = YOLO(str(BEST))
det_v1 = YOLO(str(V1_WEIGHTS))

def predict(model, paths, batch=32):
    out = {}
    for s in range(0, len(paths), batch):
        for res in model.predict([str(p) for p in paths[s:s + batch]], imgsz=IMG_SIZE, conf=min(CONF_GRID), verbose=False):
            b = res.boxes
            out[str(Path(res.path))] = {"conf": b.conf.cpu().numpy().tolist(), "xyxy": b.xyxy.cpu().numpy().tolist(),
                                        "wh": res.orig_shape[::-1]}
    return out

eval_paths = [WORK_DIR / "images" / r["split"] / r["yolo_name"] for r in items if r["split"] in ("val", "test")]
td_paths = [p for d in TEST_DATA_DIRS.values() for p in list_images(d)]
pred, pred_v1 = predict(det, eval_paths), predict(det_v1, eval_paths)
pred_td, pred_td_v1 = predict(det, td_paths), predict(det_v1, td_paths)
print(f"Predicted {len(pred)} val/test pictures and {len(pred_td)} Test_data tuna / sea frames, with v2 and v1.")

Predicted 1122 val/test pictures and 693 Test_data tuna / sea frames, with v2 and v1.


In [7]:
# --- Cell 7: Choose the cut-off on VAL. False alarms are averaged per kind, so prawn counts as much as sea or tuna. ---
def iou(a, b):
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy; u = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / u if u > 0 else 0.0

def gt_xyxy(r, wh):
    W, H = wh
    return [((xc - w / 2) * W, (yc - h / 2) * H, (xc + w / 2) * W, (yc + h / 2) * H) for xc, yc, w, h in r["boxes"]]

def outcome(r, p, conf):
    hits = [b for c, b in zip(p["conf"], p["xyxy"]) if c >= conf]
    if r["kind"] == "cetacean":
        return "found" if any(iou(h, g) >= 0.3 for h in hits for g in gt_xyxy(r, p["wh"])) else "missed"
    return "false_alarm" if hits else "quiet"

def key(r):
    return str(WORK_DIR / "images" / r["split"] / r["yolo_name"])

def scores(split, P, conf):
    rs = [r for r in items if r["split"] == split]
    pos = [outcome(r, P[key(r)], conf) == "found" for r in rs if r["kind"] == "cetacean"]
    per = {}
    for k in sorted({r["kind"] for r in rs} - {"cetacean"}):
        o = [outcome(r, P[key(r)], conf) == "false_alarm" for r in rs if r["kind"] == k]
        per[k] = (sum(o), len(o))
    found = sum(pos) / max(1, len(pos))
    fa_mean = np.mean([a / b for a, b in per.values() if b]) if per else 0.0
    return found, fa_mean, per, len(pos)

print("VAL (v2) - cetacean found vs false alarms per kind")
val_rows = []
for conf in CONF_GRID:
    found, fa, per, npos = scores("val", pred, conf)
    val_rows.append((conf, found, fa, per))
    print(f"  conf {conf:<5} found {100 * found:5.1f}% of {npos}   mean false alarm {100 * fa:5.1f}%   " +
          "  ".join(f"{k} {a}/{b}" for k, (a, b) in per.items()))
CONF = max(val_rows, key=lambda t: ((t[1] + (1 - t[2])) / 2, t[0]))[0]
print(f"\nChosen on VAL: cut-off {CONF}   (v1 used {V1['confidence_cutoff']})")

VAL (v2) - cetacean found vs false alarms per kind
  conf 0.1   found  98.3% of 413   mean false alarm   0.0%   prawn 0/27  sea 0/62  tuna 0/50
  conf 0.15  found  97.3% of 413   mean false alarm   0.0%   prawn 0/27  sea 0/62  tuna 0/50
  conf 0.2   found  94.4% of 413   mean false alarm   0.0%   prawn 0/27  sea 0/62  tuna 0/50
  conf 0.25  found  92.3% of 413   mean false alarm   0.0%   prawn 0/27  sea 0/62  tuna 0/50
  conf 0.3   found  90.3% of 413   mean false alarm   0.0%   prawn 0/27  sea 0/62  tuna 0/50
  conf 0.4   found  86.2% of 413   mean false alarm   0.0%   prawn 0/27  sea 0/62  tuna 0/50
  conf 0.5   found  81.6% of 413   mean false alarm   0.0%   prawn 0/27  sea 0/62  tuna 0/50
  conf 0.6   found  77.5% of 413   mean false alarm   0.0%   prawn 0/27  sea 0/62  tuna 0/50
  conf 0.7   found  70.0% of 413   mean false alarm   0.0%   prawn 0/27  sea 0/62  tuna 0/50

Chosen on VAL: cut-off 0.1   (v1 used 0.1)


In [8]:
# --- Cell 8: TEST - reported once, v2 at its cut-off next to v1 at its own ---
from scipy.stats import beta
def rate(k, n):
    if n == 0:
        return "n/a"
    lo = 0.0 if k == 0 else beta.ppf(0.025, k, n - k + 1); hi = 1.0 if k == n else beta.ppf(0.975, k + 1, n - k)
    return f"{k}/{n} = {100 * k / n:5.1f}% (CI {100 * lo:.0f}-{100 * hi:.0f}%)"

res = {"conf": CONF}
C1 = V1["confidence_cutoff"]
test_rs = [r for r in items if r["split"] == "test"]
print(f"TEST SPLIT          {'v2 (cut-off ' + str(CONF) + ')':<34}{'v1 (cut-off ' + str(C1) + ')':<34}")
pos = [r for r in test_rs if r["kind"] == "cetacean"]
f2 = sum(outcome(r, pred[key(r)], CONF) == "found" for r in pos); f1 = sum(outcome(r, pred_v1[key(r)], C1) == "found" for r in pos)
print(f"  cetacean found    {rate(f2, len(pos)):<34}{rate(f1, len(pos)):<34}")
res["test_cetacean_images"] = [f2, len(pos)]
for k in sorted({r["kind"] for r in test_rs} - {"cetacean"}):
    rs = [r for r in test_rs if r["kind"] == k]
    a2 = sum(outcome(r, pred[key(r)], CONF) == "false_alarm" for r in rs); a1 = sum(outcome(r, pred_v1[key(r)], C1) == "false_alarm" for r in rs)
    print(f"  {k:<6} wrongly flagged {rate(a2, len(rs)):<34}{rate(a1, len(rs)):<34}")
    res[f"test_false_alarm_{k}"] = [a2, len(rs)]
print("\nSTAGE_03/Test_data tuna / sea frames (not trained on)")
for k, d in TEST_DATA_DIRS.items():
    ps = list_images(d)
    a2 = sum(any(c >= CONF for c in pred_td[str(p)]["conf"]) for p in ps); a1 = sum(any(c >= C1 for c in pred_td_v1[str(p)]["conf"]) for p in ps)
    print(f"  {k:<6} wrongly flagged {rate(a2, len(ps)):<34}{rate(a1, len(ps)):<34}")
    res[f"test_data_false_alarm_{k}"] = [a2, len(ps)]
m = det.val(data=str(yaml_path), split="test", imgsz=IMG_SIZE, verbose=False, plots=False)
res["test_map50"] = float(m.box.map50)
print(f"\nmAP@0.5 on the test split: v2 {m.box.map50:.3f}   (v1 reported {V1['results'].get('test_map50', float('nan')):.3f} on a test split without prawns)")

TEST SPLIT          v2 (cut-off 0.1)                  v1 (cut-off 0.1)                  
  cetacean found    424/442 =  95.9% (CI 94-98%)      418/442 =  94.6% (CI 92-96%)      
  prawn  wrongly flagged 2/32 =   6.2% (CI 1-21%)          12/32 =  37.5% (CI 21-56%)        
  sea    wrongly flagged 1/60 =   1.7% (CI 0-9%)           0/60 =   0.0% (CI 0-6%)           
  tuna   wrongly flagged 1/36 =   2.8% (CI 0-15%)          0/36 =   0.0% (CI 0-10%)          

STAGE_03/Test_data tuna / sea frames (not trained on)
  tuna   wrongly flagged 0/236 =   0.0% (CI 0-2%)          4/236 =   1.7% (CI 0-4%)          
  sea    wrongly flagged 12/457 =   2.6% (CI 1-5%)         9/457 =   2.0% (CI 1-4%)          
Ultralytics 8.4.178  Python-3.14.6 torch-2.14.0+cpu CPU (AMD Ryzen 7 7435HS)
Model summary (fused): 73 layers, 3,132,707 parameters, 0 gradients, 8.6 GFLOPs
val: Fast image access  (ping: 0.0±0.0 ms, read: 530.9±58.4 MB/s, size: 203.8 KB)
val: Scanning C:\Users\bcura\AI PostGrad Project\Saqqara-M

In [9]:
# --- Cell 9: Save v2 ---
dst = OUT_DIR / "cetacean_detector_v2.pt"
shutil.copy2(BEST, dst)
bundle = {
    "weights": dst.name, "confidence_cutoff": CONF, "chosen_on": "val split (found + 1 - mean per-kind false alarm)",
    "img_size": IMG_SIZE, "finetuned_from": V1_WEIGHTS.name if FINETUNE_FROM_V1 else None,
    "prawn_rule": {"used_similarity": USED_SIMILARITY, "group_similarity": PRAWN_GROUP_SIMILARITY, "oversample": PRAWN_OVERSAMPLE},
    "files": {s: [{"source": str(r["path"]), "kind": r["kind"], "group": str(r["group"])} for r in items if r["split"] == s]
              for s in ("train", "val", "test")},
    "results": res,
    "val_table": [{"conf": c, "cetacean_found": f, "mean_false_alarm": fa} for c, f, fa, _ in val_rows],
}
with open(OUT_DIR / "cetacean_detector_v2.json", "w") as f:
    json.dump(bundle, f, indent=1)
print("Saved:", dst)
print("Saved:", OUT_DIR / "cetacean_detector_v2.json")
print("Next: in the end-to-end notebook, cell 1 now points at cetacean_detector_v2 - run it once.")

Saved: C:\Users\bcura\AI PostGrad Project\Saqqara-Marine-Anomaly-Detection\Stage_03\stage3_output\cetacean_detector_v2.pt
Saved: C:\Users\bcura\AI PostGrad Project\Saqqara-Marine-Anomaly-Detection\Stage_03\stage3_output\cetacean_detector_v2.json
Next: in the end-to-end notebook, cell 1 now points at cetacean_detector_v2 - run it once.


In [2]:
import sys; print(sys.executable)

C:\Users\bcura\anaconda3\python.exe
